# Automatic sleep staging with **Hypnos** on the Dreem Open Datasets

This notebook demonstrates using Hypnos embeddings for automated sleep staging:

1. **Download** the [Dreem Open Dataset – DOD‑O](https://arxiv.org/abs/1911.03221) (patients with
   obstructive sleep apnea, each scored by 5 sleep experts) — streaming one record at a time from
   [Zenodo](https://zenodo.org/records/15900394), *idempotently*.
2. **Embed** each night with Hypnos and pool the 1 Hz vectors into one embedding per 30 s epoch.
3. **Sleep‑stage** with a **linear probe** (logistic regression on the *frozen* embeddings),
   evaluated with **subject‑wise 5‑fold cross‑validation**.

## 1 · Install dependencies

First, install a few extras for data streaming, ML and plotting. `remotezip` lets us pull individual `.h5` records out of the big Zenodo archive over HTTP range requests.

In [ ]:
%pip install -q h5py remotezip scikit-learn matplotlib tqdm
%pip install -q -e .

import inspect
import hypnos
from hypnos.embedding import tokenize

## 2 · Configuration

DOD‑O lives in a single ~36 GB `dodo.zip` on Zenodo. `remotezip` reads the zip's central directory and fetches **only the members we ask for** via HTTP range requests. Each record is ~600 MB, so a fetch takes ~1–2 min; we therefore cache the tiny pooled
embeddings (`.npz`, a few MB/record) so re‑runs never touch the network or the model again.

Set `MAX_RECORDS` small (e.g. `3`) for a quick pass, or `None` for all 55 DOD‑O records.

In [ ]:
from pathlib import Path
import numpy as np
import torch

REPO = Path.cwd()
DATA_DIR = REPO / "data" / "dod-o"      # raw .h5 (only kept if KEEP_RAW_H5)
CACHE_DIR = REPO / "data" / "cache"     # per-record pooled embeddings (.npz)
DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

ZIP_URL = "https://zenodo.org/api/records/15900394/files/dodo.zip/content"  # DOD-O archive
MAX_RECORDS = None                       # None = all 55; set e.g. 3 for a quick smoke run
KEEP_RAW_H5 = False                      # True keeps the ~600 MB/record .h5 on disk (else stream-only)
NOTCH_FREQ = 50.0                        # DOD recorded in Europe -> 50 Hz powerline
MODEL_REPO = "joncarter/hypnos"          # public pretrained Hypnos weights on the HF Hub
SEG_SECONDS = 1800                        # tokenize each record in 30-min segments (bounds encoder memory on a full night). Lower to 900/600 if still tight.
EMBED_CHUNK = 1024                        # transformer attention chunk; peak mem ~quadratic (~2 GB @1024, ~8 GB @2048).

def pick_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

DEVICE = pick_device()

def free_memory():
    import gc
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

print("device:", DEVICE, "| cache:", CACHE_DIR)

## 3 · Open the remote archive

We open the Zenodo zip once and list its record members. Nothing large is downloaded yet — only the
zip index. Individual records are streamed later, on demand.

In [ ]:
from remotezip import RemoteZip

def is_record(name):
    base = name.rsplit("/", 1)[-1]
    return name.endswith(".h5") and "__MACOSX" not in name and not base.startswith("._")

zf = RemoteZip(ZIP_URL)                                   # central directory fetched lazily
members = sorted(n for n in zf.namelist() if is_record(n))
if MAX_RECORDS is not None:
    members = members[:MAX_RECORDS]
print(f"{len(members)} DOD-O record(s) selected (of {sum(is_record(n) for n in zf.namelist())} in archive)")
print("example member:", members[0])

## 4 · Map DOD channels → Hypnos modalities

Hypnos has 8 modalities; **DOD‑O** provides (all at 250 Hz):

| Hypnos modality | DOD‑O dataset        | present |
|-----------------|----------------------|---------|
| `eeg_c3`        | `signals/eeg/C3_M2`  | ✅ |
| `eeg_c4`        | `signals/eeg/C4_M1`  | ✅ |
| `eog_e1`        | `signals/eog/EOG1`   | ✅ |
| `eog_e2`        | `signals/eog/EOG2`   | ✅ |
| `emg_chin`      | `signals/emg/EMG`    | ✅ |
| `ecg`           | `signals/emg/ECG`    | ✅ |
| `resp_abd/thx`  | —                    | ❌ (no respiratory belts) → skipped |

DOD derivations are **already referenced** (e.g. `C3_M2`), so we skip Hypnos's EDF referencing and
feed the signals straight into the *same* preprocessing functions `preprocess_edf` uses —
`resample_signal` and `causal_preprocess_signal`. Each record's `description` attribute (JSON) gives
the per‑signal sampling rate; the expert labels are the top‑level `hypnogram` dataset (one code per
30 s epoch: `-1`=unscored, `0`=W, `1`=N1, `2`=N2, `3`=N3, `4`=REM).

In [ ]:
import json
from hypnos.data.preprocessing import resample_signal, causal_preprocess_signal

# Hypnos modality name -> DOD-O h5 dataset path.
CHANNEL_MAP = {
    "eeg_c3":   "signals/eeg/C3_M2",
    "eeg_c4":   "signals/eeg/C4_M1",
    "eog_e1":   "signals/eog/EOG1",
    "eog_e2":   "signals/eog/EOG2",
    "emg_chin": "signals/emg/EMG",
    "ecg":      "signals/emg/ECG",
}

def read_description(f) -> list:
    '''Parse the JSON `description` attr -> list of {path, fs, ...} dicts.'''
    d = f.attrs.get("description")
    if isinstance(d, bytes):
        d = d.decode()
    return json.loads(d) if d else []

def h5_to_signals(f, meta, notch_freq=NOTCH_FREQ) -> dict:
    '''Open DOD-O h5 file -> Hypnos `{modality_name: preprocessed 1-D float32}` dict.'''
    specs = {m.name: m for m in meta.modalities}
    fs_by_path = {e["path"]: int(e["fs"]) for e in read_description(f)}
    signals = {}
    for mod_name, path in CHANNEL_MAP.items():
        if path not in f or mod_name not in specs or path not in fs_by_path:
            continue                                        # absent -> skipped modality
        m = specs[mod_name]
        raw = np.asarray(f[path][:], dtype=np.float64).reshape(-1)
        sig = resample_signal(raw, fs_by_path[path], m.sample_rate)
        proc, _, _ = causal_preprocess_signal(
            sig, fs=m.sample_rate, modality=m.preprocess_modality, notch_freq=notch_freq,
        )
        signals[mod_name] = np.asarray(proc, dtype=np.float32)
    return signals

## 5 · Load the Hypnos model

In [ ]:
from hypnos.embedding import load_model, tokenize, embed

model, tokenizers, meta = load_model(MODEL_REPO, device=DEVICE)
print("Supported modalities:", [m.name for m in meta.modalities])

## 6 · Generate embeddings & pool to 30 s epochs (cached)

For each record: stream the `.h5` (or reuse a local copy / cache) → build the `signals` dict →
`tokenize` → `embed` (per‑modality `[T, 768]` at 1 Hz) → **mean over modalities** → **mean‑pool each
30 s** → `[n_epochs, 768]`, aligned to the hypnogram (unscored `-1` epochs dropped). Each record's
`(X, y)` is cached to `.npz`, so the network + model only run once per record. First pass is
~1–2 min/record.

In [ ]:
import io, time, h5py
from tqdm.auto import tqdm
from remotezip import RemoteZip

STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]   # hypnogram codes 0..4
HDF5_MAGIC = b"\x89HDF\r\n\x1a\n"

def fetch_member_bytes(member, retries=6):
    '''Stream one record from Zenodo, retrying on dropped connections. Verifies the full
    member arrived (size + HDF5 signature) and reopens a fresh RemoteZip between attempts,
    so a transient IncompleteRead/ProtocolError doesn't abort the whole run.'''
    global zf
    expected = zf.getinfo(member).file_size
    last = None
    for attempt in range(retries):
        try:
            data = zf.read(member)
            if len(data) == expected and data[:8] == HDF5_MAGIC:
                return data
            last = f"incomplete ({len(data)}/{expected} bytes)"
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
        wait = min(30, 2 ** attempt)
        print(f"  fetch {member} failed [{last}] — retry {attempt + 1}/{retries} in {wait}s")
        time.sleep(wait)
        zf = RemoteZip(ZIP_URL)                 # fresh connection for the next attempt
    raise RuntimeError(f"could not fetch {member} after {retries} attempts: {last}")

def record_features(member, verbose=False):
    '''Return (X_epochs [n,768] float32, y [n] int) for one record, cached on disk.'''
    rid = member.split("/")[-1][:-3]           # strip 'dodo/' and '.h5'
    cache = CACHE_DIR / f"{rid}.npz"
    if cache.exists():
        d = np.load(cache)
        return d["X"], d["y"]

    raw_path = DATA_DIR / f"{rid}.h5"
    if raw_path.exists():
        data = raw_path.read_bytes()
    else:
        data = fetch_member_bytes(member)      # resilient ranged fetch (retries on drop)
        if KEEP_RAW_H5:
            raw_path.write_bytes(data)

    if data[:8] != HDF5_MAGIC:                  # guard: not a valid HDF5 file -> skip
        print("skipped (not an HDF5 record):", member)
        return None

    with h5py.File(io.BytesIO(data), "r") as f:
        if verbose:
            print("  signals:", [e["path"] for e in read_description(f)])
        signals = h5_to_signals(f, meta)
        hyp = np.asarray(f["hypnogram"][:], dtype=int).reshape(-1)
    del data                                                       # free the ~600 MB raw bytes
    if verbose:
        print("  -> Hypnos modalities used:", sorted(signals))
    if not signals:
        return None

    # tokenize() chunks internally (SEG_SECONDS windows, bit-exact); embed() chunks the transformer.
    tokens, mask, ch_ids = tokenize(tokenizers, meta, signals, device=DEVICE, chunk_seconds=SEG_SECONDS)
    emb = embed(model, tokens, mask, ch_ids, meta, device=DEVICE, chunk_tokens=EMBED_CHUNK)  # {mod: [T, 768]}
    fused = np.mean(list(emb.values()), axis=0).astype(np.float32)  # [T, 768]

    n_ep = fused.shape[0] // 30
    epochs = fused[: n_ep * 30].reshape(n_ep, 30, -1).mean(axis=1)  # [n_ep, 768]
    n = min(len(epochs), len(hyp))
    epochs, hyp = epochs[:n], hyp[:n]
    keep = hyp >= 0                                                 # drop unscored epochs
    X, y = epochs[keep], hyp[keep]
    np.savez_compressed(cache, X=X, y=y)
    return X, y

X_list, y_list, groups = [], [], []
for i, member in enumerate(tqdm(members, desc="records")):
    out = record_features(member, verbose=(i == 0))
    free_memory()                              # release cache between records
    if out is None:
        continue
    Xr, yr = out
    X_list.append(Xr); y_list.append(yr); groups += [i] * len(yr)

X = np.concatenate(X_list); y = np.concatenate(y_list); groups = np.asarray(groups)
print("\nX:", X.shape, "| y:", y.shape, "| records:", len(set(groups)))
print("epochs per stage:", {STAGE_NAMES[k]: int((y == k).sum()) for k in range(5)})

## 7 · Linear probe with subject‑wise cross‑validation

A standardizer + multinomial logistic regression — a **linear probe** on the frozen embeddings — evaluated with `GroupKFold` so every fold's test subjects are unseen in training (avoids the optimistic leakage of splitting epochs within a night).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, cohen_kappa_score, confusion_matrix

n_splits = min(5, len(set(groups)))
gkf = GroupKFold(n_splits=n_splits)
y_pred = np.empty_like(y)

print(f"subject-wise {n_splits}-fold CV\n")
for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000),
    )
    clf.fit(X[tr], y[tr])
    y_pred[te] = clf.predict(X[te])
    print(f"  fold {fold}: bal_acc={balanced_accuracy_score(y[te], y_pred[te]):.3f}  "
          f"macroF1={f1_score(y[te], y_pred[te], average='macro'):.3f}  "
          f"kappa={cohen_kappa_score(y[te], y_pred[te]):.3f}")

## 8 · Overall metrics

In [ ]:
print("=== overall (held-out predictions, pooled across folds) ===")
print(f"balanced accuracy : {balanced_accuracy_score(y, y_pred):.3f}")
print(f"macro F1          : {f1_score(y, y_pred, average='macro'):.3f}")
print(f"Cohen's kappa     : {cohen_kappa_score(y, y_pred):.3f}")

## 9 · Confusion matrix & hypnogram

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y, y_pred, labels=range(5)).astype(float)
cmn = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(5)); ax.set_xticklabels(STAGE_NAMES)
ax.set_yticks(range(5)); ax.set_yticklabels(STAGE_NAMES)
ax.set_xlabel("predicted"); ax.set_ylabel("expert consensus")
for i in range(5):
    for j in range(5):
        ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center",
                color="white" if cmn[i, j] > 0.5 else "black", fontsize=8)
ax.set_title("Sleep-stage confusion (row-normalized)")
fig.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

In [ ]:
# Predicted vs expert hypnogram for one held-out record.
rec = sorted(set(groups))[-1]
m_rec = groups == rec
order = [0, 4, 1, 2, 3]                       # W, REM, N1, N2, N3 (classic top->bottom)
pos = {s: i for i, s in enumerate(order)}
t = np.arange(m_rec.sum()) / 2 / 60           # 30 s epochs -> hours

fig, ax = plt.subplots(figsize=(12, 3))
ax.step(t, [pos[s] for s in y[m_rec]],      where="post", lw=1.3, label="expert")
ax.step(t, [pos[s] for s in y_pred[m_rec]], where="post", lw=1.0, alpha=0.7, label="predicted")
ax.set_yticks(range(5)); ax.set_yticklabels([STAGE_NAMES[s] for s in order])
ax.invert_yaxis()                             # W at top, N3 at bottom
ax.set_xlabel("time (hours)")
ax.set_title(f"Hypnogram — held-out record {members[rec].split('/')[-1][:8]}")
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()